# Week 2 results: can a regime forecast improve the strategies?

1. Train a classifier to predict whether **next month** will be stressed (SPY volatility above
   the 80th percentile of its past year), using only information available today.
2. Evaluate it honestly: chronological folds with a 21-day gap, compared with "always calm".
3. Use its walk-forward probabilities to switch strategies into minimum variance when stress is
   forecast, and compare with the static strategies and a simple volatility rule.

**Not investment advice.**

In [ ]:
import logging

import pandas as pd

from gpl import backtest as bt
from gpl import regime as rg
from gpl.data import clean_prices, get_prices, get_risk_free, load_universe
from gpl.metrics import drawdown_series, summary_table
from gpl.plots import drawdown_chart, equity_chart, regime_chart

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
pd.options.display.float_format = "{:,.3f}".format

cfg = load_universe()
d, R = cfg["defaults"], cfg["regime"]

## 1. Regime data (SPY, AGG, VIX from 2005)

In [ ]:
regime_px = clean_prices(get_prices([R["equity"], R["bond"]], R["start"], d["end"]))
vix = get_prices(["^VIX"], R["start"], d["end"])["^VIX"]
features = rg.make_features(regime_px, vix, equity=R["equity"], bond=R["bond"])
labels = rg.make_labels(
    regime_px[R["equity"]].pct_change().iloc[1:], R["horizon"], R["quantile"], R["window"]
)
print(f"Sample: {regime_px.index[0].date()} to {regime_px.index[-1].date()}")
print(f"Share of days labelled stressed: {labels.mean():.1%}")
features.describe().T

## 2. Out-of-sample evaluation (5 chronological folds, 21-day gap)

In [ ]:
reports = {
    m: rg.evaluate_model(features, labels, model=m, horizon=R["horizon"], threshold=R["threshold"])
    for m in ["logistic", "random_forest"]
}


def overall(report: dict) -> pd.Series:
    c = report["confusion"].to_numpy()
    tn, fp, fn, tp = c.ravel()
    return pd.Series(
        {
            "precision": tp / (tp + fp),
            "recall": tp / (tp + fn),
            "accuracy": (tp + tn) / c.sum(),
            "always_calm_accuracy": (tn + fp) / c.sum(),
            "stressed_share": (tp + fn) / c.sum(),
        }
    )


pd.DataFrame({m: overall(r) for m, r in reports.items()}).T

In [ ]:
for m, r in reports.items():
    print(m)
    display(
        r["folds"][
            [
                "test_start",
                "test_end",
                "stressed_share",
                "precision",
                "recall",
                "accuracy",
                "baseline_accuracy",
            ]
        ]
    )
    display(r["confusion"])

**Reading this:** recall is the share of truly stressed days the model flagged. Precision is
how often a "stressed" call was right. "Always calm" has higher accuracy simply because calm is
the majority class. It never catches a single stressed month, which is the thing we care about.

## 3. Walk-forward probabilities (what an investor would have seen live)

In [ ]:
p_stressed = rg.walk_forward_probabilities(
    features,
    labels,
    model=R["model"],
    refit=R["refit"],
    min_train=R["min_train"],
    horizon=R["horizon"],
)
naive = rg.naive_stress_signal(features, R["quantile"], R["window"])
regime_chart(p_stressed, R["threshold"], labels).show()

## 4. Regime-aware strategies vs static ones (Sep 2016 to Sep 2026, 10 bps costs)

In [ ]:
prices = clean_prices(get_prices(cfg["assets"], d["start"], d["end"]), max_ffill=d["max_ffill"])
rf = get_risk_free(d["start"], d["end"])
mw = d["max_weight"]
static = {
    "Equal weight": bt.equal_weight,
    "Max Sharpe": bt.max_sharpe_strategy(rf=rf, max_weight=mw),
    "Risk parity": bt.risk_parity_strategy(max_weight=mw),
}
defensive = bt.min_variance_strategy(max_weight=mw)
strategies = {
    **static,
    **{
        f"{name} + regime": bt.regime_switch_strategy(s, defensive, p_stressed, R["threshold"])
        for name, s in static.items()
    },
    **{
        f"{name} + vol rule": bt.regime_switch_strategy(s, defensive, naive, 0.5)
        for name, s in static.items()
    },
    "Min variance": defensive,
    "60/40": bt.fixed_weights(cfg["benchmarks"]["sixty_forty"]),
    "SPY": bt.fixed_weights(cfg["benchmarks"]["spy"]),
}
run = dict(rebalance=d["rebalance"], lookback=d["lookback"], cost_bps=d["cost_bps"])
results = {name: bt.run_backtest(prices, s, **run) for name, s in strategies.items()}
returns = pd.DataFrame({n: r.returns for n, r in results.items()})

rebalance_dates = results["Equal weight"].weights.index
known = p_stressed.reindex(rebalance_dates, method="ffill")
print(
    f"Months de-risked by the model: {(known >= R['threshold']).mean():.0%}; "
    f"by the vol rule: {(naive.reindex(rebalance_dates, method='ffill') >= 0.5).mean():.0%}"
)

table = summary_table(returns, rf=rf, level=d["var_level"])
table["annual_turnover"] = pd.Series(
    {n: r.turnover.iloc[1:].mean() * 12 for n, r in results.items()}
)
table[
    [
        "cagr",
        "ann_vol",
        "sharpe",
        "sortino",
        "max_drawdown",
        "expected_shortfall",
        "annual_turnover",
    ]
].sort_values("sharpe", ascending=False)

In [ ]:
show = [
    "Equal weight",
    "Equal weight + regime",
    "Equal weight + vol rule",
    "Risk parity",
    "Risk parity + regime",
    "Min variance",
    "60/40",
    "SPY",
]
equity_chart(
    pd.DataFrame({n: results[n].equity for n in show}), title="Growth of $1: static vs regime-aware"
).show()
drawdown_chart(drawdown_series(returns[show])).show()

In [ ]:
# Drawdowns in the two big stress episodes of the backtest window.
episodes = {
    "COVID (Feb-Apr 2020)": ("2020-02-01", "2020-04-30"),
    "2022 rates shock": ("2022-01-01", "2022-10-31"),
}
pd.DataFrame({ep: drawdown_series(returns.loc[a:b]).min() for ep, (a, b) in episodes.items()}).loc[
    [
        "Equal weight",
        "Equal weight + regime",
        "Equal weight + vol rule",
        "Risk parity",
        "Risk parity + regime",
        "Min variance",
        "60/40",
        "SPY",
    ]
]

## Limitations

- **One forecast target:** stress is defined by SPY volatility. GCC-specific shocks (oil, local
  politics) that don't move SPY are invisible to this model.
- **Few independent episodes:** 20 years of daily rows sounds like a lot, but there are only a
  handful of separate stress periods. Overlapping 21-day labels make neighbouring rows nearly
  identical.
- **Threshold choice:** 0.5 with class-balanced models flags stress often. A different threshold
  trades recall for precision. Tuning it on this same sample would be overfitting.
- **Switching to min variance** is one of many possible de-risking rules.